In [24]:
#https://huggingface.co/docs/transformers/tasks/token_classification


In [25]:
from datasets import load_dataset
db=load_dataset('wnut_17')
db

Using the latest cached version of the dataset since wnut_17 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wnut_17' at C:\Users\drnwe\.cache\huggingface\datasets\wnut_17\wnut_17\1.0.0\9a631b654e7ed17a61ab0aaf6901ca1654b99a88 (last modified on Tue Jan  9 11:31:15 2024).


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1287
    })
})

In [26]:
db['train'][0]

{'id': '0',
 'tokens': ['@paulwalk',
  'It',
  "'s",
  'the',
  'view',
  'from',
  'where',
  'I',
  "'m",
  'living',
  'for',
  'two',
  'weeks',
  '.',
  'Empire',
  'State',
  'Building',
  '=',
  'ESB',
  '.',
  'Pretty',
  'bad',
  'storm',
  'here',
  'last',
  'evening',
  '.'],
 'ner_tags': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  7,
  8,
  8,
  0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [27]:
label_list=db['train'].features["ner_tags"].feature.names
label_list

['O',
 'B-corporation',
 'I-corporation',
 'B-creative-work',
 'I-creative-work',
 'B-group',
 'I-group',
 'B-location',
 'I-location',
 'B-person',
 'I-person',
 'B-product',
 'I-product']

In [28]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [29]:
example=db['train'][0]
example

{'id': '0',
 'tokens': ['@paulwalk',
  'It',
  "'s",
  'the',
  'view',
  'from',
  'where',
  'I',
  "'m",
  'living',
  'for',
  'two',
  'weeks',
  '.',
  'Empire',
  'State',
  'Building',
  '=',
  'ESB',
  '.',
  'Pretty',
  'bad',
  'storm',
  'here',
  'last',
  'evening',
  '.'],
 'ner_tags': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  7,
  8,
  8,
  0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [30]:
tokenized_inputs=tokenizer(example['tokens'],is_split_into_words=True)
tokenized_inputs

{'input_ids': [101, 1030, 2703, 17122, 2009, 1005, 1055, 1996, 3193, 2013, 2073, 1045, 1005, 1049, 2542, 2005, 2048, 3134, 1012, 3400, 2110, 2311, 1027, 9686, 2497, 1012, 3492, 2919, 4040, 2182, 2197, 3944, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [31]:
def tokenize_and_align_labels(examples):
    tokenized_inputs=tokenizer(examples['tokens'],truncation=True,is_split_into_words=True)
    labels=[]
    for i,label in enumerate(examples['ner_tags']):
        word_ids=tokenized_inputs.word_ids(batch_index=i) 
        previous_word_idx=None
        label_ids=[]
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx!=previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
        labels.append(label_ids)
    tokenized_inputs['labels']=labels
    return tokenized_inputs
        

In [32]:
tokenized_db=db.map(tokenize_and_align_labels,batched=True)
tokenized_db

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1287
    })
})

In [33]:
from transformers import DataCollatorForTokenClassification
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

In [34]:
import evaluate
seqeval=evaluate.load('seqeval')

In [35]:
import numpy as np
labels=[label_list[x] for x in example['ner_tags']]
labels

['O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-location',
 'I-location',
 'I-location',
 'O',
 'B-location',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O']

In [36]:
def compute_metrics(p):
    predictions,labels=p
    predictions=np.argmax(predictions,axis=2)
    true_predictions=[
        [label_list[p] for (p,l) in zip(prediction,label) if l!=-100]
        for prediction,label in zip(predictions,labels)
    ]
    true_labels=[
        [label_list[l] for (p,l) in zip(prediction,label) if l!=-100]
        for prediction,label in zip(predictions,labels)
    ]
    results=seqeval.compute(predictions=true_predictions,references=true_labels)
    return {
        "precision":results['overall_precision'],
        "recall":results['overall_recall'],
        "f1":results['overall_f1'],
        "accuracy":results['overall_accuracy'],
    }

In [37]:
id2label = {
    0: "O",
    1: "B-corporation",
    2: "I-corporation",
    3: "B-creative-work",
    4: "I-creative-work",
    5: "B-group",
    6: "I-group",
    7: "B-location",
    8: "I-location",
    9: "B-person",
    10: "I-person",
    11: "B-product",
    12: "I-product",
}
label2id={k:v for v,k in id2label.items()}

In [38]:
id2label

{0: 'O',
 1: 'B-corporation',
 2: 'I-corporation',
 3: 'B-creative-work',
 4: 'I-creative-work',
 5: 'B-group',
 6: 'I-group',
 7: 'B-location',
 8: 'I-location',
 9: 'B-person',
 10: 'I-person',
 11: 'B-product',
 12: 'I-product'}

In [39]:
label2id

{'O': 0,
 'B-corporation': 1,
 'I-corporation': 2,
 'B-creative-work': 3,
 'I-creative-work': 4,
 'B-group': 5,
 'I-group': 6,
 'B-location': 7,
 'I-location': 8,
 'B-person': 9,
 'I-person': 10,
 'B-product': 11,
 'I-product': 12}

In [40]:
from transformers import AutoModelForTokenClassification,TrainingArguments,Trainer
model=AutoModelForTokenClassification.from_pretrained(
    'distilbert-base-uncased',num_labels=13,id2label=id2label,label2id=label2id
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
training_args=TrainingArguments(
    output_dir='my_ner_model',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_db['train'],
    eval_dataset=tokenized_db['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.367030,0.480249,0.114413,0.184800,0.930262
2,No log,0.373867,0.493151,0.196137,0.280652,0.934956
3,0.178700,0.383784,0.522293,0.203071,0.292439,0.936259


C:\Users\drnwe\anaconda3\envs\galaxie\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Checkpoint destination directory my_ner_model\checkpoint-213 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory my_ner_model\checkpoint-426 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory my_ner_model\checkpoint-639 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=639, training_loss=0.15879533436376725, metrics={'train_runtime': 732.5128, 'train_samples_per_second': 13.9, 'train_steps_per_second': 0.872, 'total_flos': 137554031124240.0, 'train_loss': 0.15879533436376725, 'epoch': 3.0})

In [43]:
import pickle
with open('ner_custom_pickle.pkl', 'wb') as f:
    pickle.dump(model, f)
print("done")

done


In [44]:
from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
filepath='my_ner_model/checkpoint-639'

# Named entity recognition pipeline, passing in a specific model and tokenizer
#model = AutoModelForTokenClassification.from_pretrained("dbmdz/bert-large-cased-finetuned-conll03-english")
model=AutoModelForTokenClassification.from_pretrained(filepath)
tokenizer = AutoTokenizer.from_pretrained(filepath)
recognizer = pipeline("ner", model=model, tokenizer=tokenizer)
input="Hello, my name is Joe and I live in Los Angeles"
result=recognizer(input)
for x in range(len(result)):
    one_row=result[x]
    entity=input[one_row['start']:one_row['end']]
    entity_name=one_row['entity']
    print(f"NER = {entity} and types= {entity_name}")

NER = Joe and types= B-person
NER = Los and types= B-location
NER = Angeles and types= B-location


In [50]:
#resume
import sys, fitz
fname = 'data/Alice Clark CV.pdf'
doc = fitz.open(fname)
all_text=""

for page_number in range(doc.page_count):
        # Select the current page
        page = doc[page_number]

        # Extract text from the page
        text = page.get_text()

        # Print or process the extracted text
        #print(f"Page {page_number + 1}:\n{text}\n")
        all_text+=" ".join(text.split('\n'))
print(all_text)

Alice Clark  AI / Machine Learning    Delhi, India Email me on Indeed  •  20+ years of experience in data handling, design, and development  •  Data Warehouse: Data analysis, star/snow flake scema data modelling and design specific to  data warehousing and business intelligence  •  Database: Experience in database designing, scalability, back-up and recovery, writing and  optimizing SQL code and Stored Procedures, creating functions, views, triggers and indexes.  Cloud platform: Worked on Microsoft Azure cloud services like Document DB, SQL Azure,  Stream Analytics, Event hub, Power BI, Web Job, Web App, Power BI, Azure data lake  analytics(U-SQL)  Willing to relocate anywhere    WORK EXPERIENCE  Software Engineer  Microsoft – Bangalore, Karnataka  January 2000 to Present  1. Microsoft Rewards Live dashboards:  Description: - Microsoft rewards is loyalty program that rewards Users for browsing and shopping  online. Microsoft Rewards members can earn points when searching with Bing, bro

In [52]:
result=recognizer(all_text)
ner_list=[]
for x in range(len(result)):
    one_row=result[x]
    entity=all_text[one_row['start']:one_row['end']]
    entity_name=one_row['entity']
    print(f"NER = {entity} and types= {entity_name}")
    ner_list.append(entity)

NER = Alice and types= B-person
NER = Delhi and types= B-location
NER = India and types= B-location
NER = Microsoft and types= B-corporation
NER = Azure and types= B-corporation
NER = Microsoft and types= B-corporation
NER = Bangalore and types= B-location
NER = Karnataka and types= B-location
NER = Microsoft and types= B-corporation
NER = Microsoft and types= B-corporation
NER = Microsoft and types= B-corporation
NER = Bing and types= B-corporation
NER = Microsoft and types= B-corporation
NER = Edge and types= B-corporation
NER = Xbox and types= B-corporation
NER = Windows and types= B-corporation
NER = Microsoft and types= B-corporation
NER = Microsoft and types= B-corporation
NER = rewards and types= B-corporation
NER = Indian and types= B-location
NER = Mumbai and types= B-location


In [53]:
ner_list

['Alice',
 'Delhi',
 'India',
 'Microsoft',
 'Azure',
 'Microsoft',
 'Bangalore',
 'Karnataka',
 'Microsoft',
 'Microsoft',
 'Microsoft',
 'Bing',
 'Microsoft',
 'Edge',
 'Xbox',
 'Windows',
 'Microsoft',
 'Microsoft',
 'rewards',
 'Indian',
 'Mumbai']

In [54]:
#remove NER from text
for x in ner_list:
    all_text=all_text.replace(x," ")
all_text=" ".join(all_text.split(' '))
all_text


'  Clark  AI / Machine Learning     ,   Email me on Indeed  •  20+ years of experience in data handling, design, and development  •  Data Warehouse: Data analysis, star/snow flake scema data modelling and design specific to  data warehousing and business intelligence  •  Database: Experience in database designing, scalability, back-up and recovery, writing and  optimizing SQL code and Stored Procedures, creating functions, views, triggers and indexes.  Cloud platform: Worked on     cloud services like Document DB, SQL  ,  Stream Analytics, Event hub, Power BI, Web Job, Web App, Power BI,   data lake  analytics(U-SQL)  Willing to relocate anywhere    WORK EXPERIENCE  Software Engineer    –  ,    January 2000 to Present  1.   Rewards Live dashboards:  Description: -     is loyalty program that   Users for browsing and shopping  online.   Rewards members can earn points when searching with  , browsing with      and making purchases at the   Store, the   Store and the    Store. Plus, user 